# Phase 4: Data Cleaning & Preprocessing

## Objective

Implement the cleaning strategy defined in Phase 3.

Goals:

- Standardize missing values
- Correct data types
- Investigate leakage-risk features
- Prepare data for feature engineering and modeling
- Create a reproducible preprocessing workflow

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
train_df = pd.read_csv("../data/raw/cell2celltrain.csv")

print(train_df.shape)

(51047, 58)


In [3]:
df = train_df.copy()

In [4]:
missing = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": df.isna().mean() * 100
})

missing = missing[missing["Missing_Count"] > 0]

missing.sort_values(
    by="Missing_Percentage",
    ascending=False
)

,Missing_Count,Missing_Percentage
AgeHH1,909,1.780712
AgeHH2,909,1.780712
PercChangeMinutes,367,0.718945
PercChangeRevenues,367,0.718945
MonthlyRevenue,156,0.305601
MonthlyMinutes,156,0.305601
TotalRecurringCharge,156,0.305601
DirectorAssistedCalls,156,0.305601
OverageMinutes,156,0.305601
RoamingCalls,156,0.305601


## 4.2 Missing Value Standardization

In [5]:
# Create working copy
df_clean = df.copy()

# Hidden missing values
df_clean["AgeHH1"] = df_clean["AgeHH1"].replace(0, np.nan)
df_clean["AgeHH2"] = df_clean["AgeHH2"].replace(0, np.nan)

# Encoded missing values
df_clean["HandsetPrice"] = df_clean["HandsetPrice"].replace("Unknown", np.nan)

In [6]:
df_clean[["AgeHH1", "AgeHH2", "HandsetPrice"]].isna().mean() * 100

AgeHH1          29.043822
AgeHH2          52.884597
HandsetPrice    56.775129
dtype: float64

In [7]:
# Missing flag for AgeHH2

df_clean["AgeHH2_Missing"] = df_clean["AgeHH2"].isna()

pd.crosstab(
    df_clean["AgeHH2_Missing"],
    df_clean["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
AgeHH2_Missing,,
False,71.664380,28.335620
True,70.751222,29.248778


In [8]:
df_clean["HandsetPrice_Missing"] = df_clean["HandsetPrice"].isna()

pd.crosstab(
    df_clean["HandsetPrice_Missing"],
    df_clean["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
HandsetPrice_Missing,,
False,72.961704,27.038296
True,69.826099,30.173901


## Missingness Signal Investigation

Before performing imputation, we investigated whether missing values themselves contain predictive information.

### AgeHH2

Churn rate comparison:

| AgeHH2 Status | Churn Rate |
|---------------|------------|
| Available | 28.34% |
| Missing | 29.25% |

Observation:

- Difference is approximately 0.91%.
- Missingness does not appear to be strongly associated with churn.

Decision:

- Retain the feature.
- Missingness flag not required at this stage.

---

### HandsetPrice

Churn rate comparison:

| HandsetPrice Status | Churn Rate |
|---------------------|------------|
| Available | 27.04% |
| Missing | 30.17% |

Observation:

- Difference is approximately 3.13%.
- Missingness appears to carry business information.

Decision:

- Retain HandsetPrice.
- Create a missingness indicator feature.
- Preserve the signal during preprocessing.

In [9]:
df_clean[["AgeHH1", "AgeHH2"]].describe()

,AgeHH1,AgeHH2
count,36221.000000,24051.000000
mean,43.379007,44.078209
std,12.385891,13.527522
min,18.000000,18.000000
25%,34.000000,34.000000
50%,42.000000,44.000000
75%,52.000000,52.000000
max,99.000000,99.000000


## Age Feature Imputation Strategy

After converting encoded missing values (0) to NaN, the age distributions were reviewed.

### Findings

| Feature | Mean | Median |
|----------|--------|--------|
| AgeHH1 | 43.38 | 42 |
| AgeHH2 | 44.08 | 44 |

Observations:

- Mean and median are very similar.
- No strong evidence of severe skewness.
- Age values fall within a realistic range.

### Decision

Median imputation will be used for both age variables.

### Rationale

- Robust to potential outliers.
- Common industry practice for demographic variables.
- Suitable for production pipelines.

In [10]:
df_clean[["AgeHH1", "AgeHH2"]].isna().sum()

AgeHH1    14826
AgeHH2    26996
dtype: int64

In [11]:
df_clean["AgeHH1"].median()
df_clean["AgeHH2"].median()

44.0

In [12]:
print(df_clean["AgeHH1"].isna().sum())
print(df_clean["AgeHH2"].isna().sum())

14826
26996


In [13]:
# AgeHH1
df_clean["AgeHH1"] = df_clean["AgeHH1"].fillna(
    df_clean["AgeHH1"].median()
)

# AgeHH2
df_clean["AgeHH2"] = df_clean["AgeHH2"].fillna(
    df_clean["AgeHH2"].median()
)

print(df_clean["AgeHH1"].isna().sum())
print(df_clean["AgeHH2"].isna().sum())

0
0


## Age Feature Cleaning

- Converted encoded missing values (0) to NaN.
- Applied median imputation.
- Validated that no missing values remain.

Result:
- AgeHH1 missing = 0
- AgeHH2 missing = 0

In [14]:
df_clean["HandsetPrice"].value_counts(dropna=False).head(20)

HandsetPrice
NaN    28982
30      7328
150     4115
130     2105
80      1960
10      1928
60      1776
200     1266
100     1235
40       249
400       46
250       20
300       13
180       10
500        8
240        6
Name: count, dtype: int64

In [15]:
df_clean["HandsetPrice"].dtype

dtype('O')

## HandsetPrice Cleaning

- Converted feature to numeric format.
- Created missing value indicator.
- Retained missingness signal.
- Applied median imputation.

Reason:
Missing handset price information showed a higher churn rate and may contain predictive value.

In [16]:
df_clean["HandsetPrice_Missing"] = (
    df_clean["HandsetPrice"].isna().astype(int)
)

In [17]:
df_clean["HandsetPrice"] = pd.to_numeric(
    df_clean["HandsetPrice"],
    errors="coerce"
)

In [18]:
df_clean["HandsetPrice"].median()

60.0

## HandsetPrice Cleaning

- Created missing value indicator.
- Converted feature to numeric.
- Imputed missing values using median (60).
- Retained missingness information through a separate flag feature.

In [19]:
df_clean["HandsetPrice"] = df_clean["HandsetPrice"].fillna(
    df_clean["HandsetPrice"].median()
)

In [20]:
df_clean["HandsetPrice"].isna().sum()

np.int64(0)

In [21]:
df_clean["MaritalStatus"].value_counts(dropna=False)

MaritalStatus
Unknown    19700
Yes        18651
No         12696
Name: count, dtype: int64

In [22]:
df_clean["Homeownership"].value_counts(dropna=False)

Homeownership
Known      33987
Unknown    17060
Name: count, dtype: int64

## MaritalStatus and Homeownership

- Retained "Unknown" as a valid category.
- No imputation performed.
- Treated as business-defined categorical values rather than missing data.

In [23]:
structured_cols = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls"
]

df_clean[structured_cols].isna().sum()

MonthlyRevenue           156
MonthlyMinutes           156
TotalRecurringCharge     156
DirectorAssistedCalls    156
OverageMinutes           156
RoamingCalls             156
dtype: int64

In [24]:
df_clean[df_clean["MonthlyRevenue"].isna()][structured_cols].head()

,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls
122,NaN,NaN,NaN,NaN,NaN,NaN
126,NaN,NaN,NaN,NaN,NaN,NaN
925,NaN,NaN,NaN,NaN,NaN,NaN
1454,NaN,NaN,NaN,NaN,NaN,NaN
2228,NaN,NaN,NaN,NaN,NaN,NaN


## Structured Missingness Investigation

A group of 156 customers exhibited missing values across the same usage and revenue-related variables.

Observation:
- Missingness occurs in an identical pattern.
- Indicates structured missingness rather than random data loss.

Decision:
- Investigate predictive impact before selecting an imputation strategy.

In [25]:
df_clean["RevenueGroup_Missing"] = (
    df_clean["MonthlyRevenue"].isna()
)

pd.crosstab(
    df_clean["RevenueGroup_Missing"],
    df_clean["Churn"],
    normalize="index"
) * 100

Churn,No,Yes
RevenueGroup_Missing,,
False,71.230669,28.769331
True,55.128205,44.871795


In [26]:
df_clean[
    [
        "MonthlyRevenue",
        "MonthlyMinutes",
        "TotalRecurringCharge",
        "DirectorAssistedCalls",
        "OverageMinutes",
        "RoamingCalls"
    ]
].describe()

,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls
count,50891.000000,50891.000000,50891.000000,50891.000000,50891.000000,50891.000000
mean,58.834492,525.653416,46.830088,0.895229,40.027785,1.236244
std,44.507336,529.871063,23.848871,2.228546,96.588076,9.818294
min,-6.170000,0.000000,-11.000000,0.000000,0.000000,0.000000
25%,33.610000,158.000000,30.000000,0.000000,0.000000,0.000000
50%,48.460000,366.000000,45.000000,0.250000,3.000000,0.000000
75%,71.065000,723.000000,60.000000,0.990000,41.000000,0.300000
max,1223.380000,7359.000000,400.000000,159.390000,4321.000000,1112.400000


## Structured Missingness Treatment

- Missingness affected 156 customers across multiple revenue and usage features.
- Missing customers showed significantly higher churn rates.
- Missingness information was preserved using an indicator feature.
- Median imputation was selected due to skewed distributions.

In [27]:
structured_cols = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls"
]

df_clean["RevenueGroup_Missing"] = (
    df_clean["MonthlyRevenue"].isna().astype(int)
)

In [28]:
for col in structured_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

In [29]:
df_clean[structured_cols].isna().sum()

MonthlyRevenue           0
MonthlyMinutes           0
TotalRecurringCharge     0
DirectorAssistedCalls    0
OverageMinutes           0
RoamingCalls             0
dtype: int64

In [30]:
remaining_cols = [
    "ServiceArea",
    "CurrentEquipmentDays",
    "Handsets",
    "HandsetModels"
]

df_clean[remaining_cols].isna().sum()

ServiceArea             24
CurrentEquipmentDays     1
Handsets                 1
HandsetModels            1
dtype: int64

In [31]:
df_clean[remaining_cols].dtypes

ServiceArea              object
CurrentEquipmentDays    float64
Handsets                float64
HandsetModels           float64
dtype: object

## Low-Missingness Features

- ServiceArea missing values were assigned to a new category: "Unknown".
- CurrentEquipmentDays, Handsets, and HandsetModels were imputed using median values.
- Missingness was negligible and did not justify record removal.

In [32]:
df_clean.isna().sum().sort_values(ascending=False).head(10)

PercChangeMinutes       367
PercChangeRevenues      367
ServiceArea              24
CurrentEquipmentDays      1
HandsetModels             1
Handsets                  1
RespondsToMailOffers      0
HasCreditCard             0
OwnsComputer              0
NonUSTravel               0
dtype: int64

In [33]:
df_clean[
    ["PercChangeMinutes", "PercChangeRevenues"]
].describe()

,PercChangeMinutes,PercChangeRevenues
count,50680.000000,50680.000000
mean,-11.547908,-1.191985
std,257.514772,39.574915
min,-3875.000000,-1107.700000
25%,-83.000000,-7.100000
50%,-5.000000,-0.300000
75%,66.000000,1.600000
max,5192.000000,2483.500000


In [34]:
df_clean[
    ["PercChangeMinutes", "PercChangeRevenues"]
].head()

,PercChangeMinutes,PercChangeRevenues
0,-157.0,-19.0
1,-4.0,0.0
2,-2.0,0.0
3,157.0,8.1
4,0.0,-0.2


In [35]:
# ServiceArea
df_clean["ServiceArea"] = df_clean["ServiceArea"].fillna("Unknown")

# Very low missing numeric features
for col in [
    "CurrentEquipmentDays",
    "Handsets",
    "HandsetModels"
]:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

## Low Missingness Features

- ServiceArea missing values assigned to "Unknown".
- CurrentEquipmentDays, Handsets, and HandsetModels imputed using median values.
- Missingness levels were negligible and did not warrant record removal.

In [36]:
df_clean[
    ["PercChangeMinutes", "PercChangeRevenues"]
].describe()

,PercChangeMinutes,PercChangeRevenues
count,50680.000000,50680.000000
mean,-11.547908,-1.191985
std,257.514772,39.574915
min,-3875.000000,-1107.700000
25%,-83.000000,-7.100000
50%,-5.000000,-0.300000
75%,66.000000,1.600000
max,5192.000000,2483.500000


## Percentage Change Features

- Missingness was below 1%.
- Distributions contained significant outliers.
- Median imputation was selected instead of mean imputation.

In [37]:
for col in [
    "PercChangeMinutes",
    "PercChangeRevenues"
]:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

In [38]:
df_clean.isna().sum().sort_values(ascending=False).head(20)

CustomerID                   0
AgeHH2                       0
HandsetRefurbished           0
HandsetWebCapable            0
TruckOwner                   0
RVOwner                      0
Homeownership                0
BuysViaMailOrder             0
RespondsToMailOffers         0
OptOutMailings               0
NonUSTravel                  0
OwnsComputer                 0
HasCreditCard                0
RetentionCalls               0
RetentionOffersAccepted      0
NewCellphoneUser             0
NotNewCellphoneUser          0
ReferralsMadeBySubscriber    0
IncomeGroup                  0
OwnsMotorcycle               0
dtype: int64

## Missing Value Treatment Completed

All identified missing-value issues were addressed.

Key actions:
- Standardized hidden and encoded missing values.
- Applied median imputation for numeric features.
- Preserved informative missingness through indicator variables where appropriate.
- Retained valid business categories such as "Unknown".

Validation:
- No missing values remain in the dataset.

In [39]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51047 entries, 0 to 51046
Data columns (total 61 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CustomerID                 51047 non-null  int64  
 1   Churn                      51047 non-null  object 
 2   MonthlyRevenue             51047 non-null  float64
 3   MonthlyMinutes             51047 non-null  float64
 4   TotalRecurringCharge       51047 non-null  float64
 5   DirectorAssistedCalls      51047 non-null  float64
 6   OverageMinutes             51047 non-null  float64
 7   RoamingCalls               51047 non-null  float64
 8   PercChangeMinutes          51047 non-null  float64
 9   PercChangeRevenues         51047 non-null  float64
 10  DroppedCalls               51047 non-null  float64
 11  BlockedCalls               51047 non-null  float64
 12  UnansweredCalls            51047 non-null  float64
 13  CustomerCareCalls          51047 non-null  flo

In [40]:
leakage_cols = [
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

for col in leakage_cols:
    print("\n" + "="*50)
    print(col)
    print(pd.crosstab(df_clean[col], df_clean["Churn"]))


RetentionCalls
Churn              No    Yes
RetentionCalls              
0               35377  13925
1                 885    724
2                  67     53
3                   7      7
4                   0      2

RetentionOffersAccepted
Churn                       No    Yes
RetentionOffersAccepted              
0                        35817  14349
1                          494    343
2                           20     16
3                            5      3

MadeCallToRetentionTeam
Churn                       No    Yes
MadeCallToRetentionTeam              
No                       35377  13925
Yes                        959    786


## Leakage Risk Assessment

Features reviewed:

- RetentionCalls
- RetentionOffersAccepted
- MadeCallToRetentionTeam

Findings:
- Customers associated with retention activities exhibit higher churn rates.
- This indicates strong association but does not confirm target leakage.

Decision:
- Retain features for now.
- Reassess during modeling and feature importance analysis.

In [41]:
categorical_cols = df_clean.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

numerical_cols = df_clean.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical:", len(categorical_cols))
print("Numerical:", len(numerical_cols))

print("\nCategorical Columns:")
print(categorical_cols)

Categorical: 23
Numerical: 38

Categorical Columns:
['Churn', 'ServiceArea', 'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'Homeownership', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 'NotNewCellphoneUser', 'OwnsMotorcycle', 'MadeCallToRetentionTeam', 'CreditRating', 'PrizmCode', 'Occupation', 'MaritalStatus', 'AgeHH2_Missing']


In [42]:
for col in [
    "ServiceArea",
    "CreditRating",
    "PrizmCode",
    "Occupation"
]:
    print("\n" + "="*50)
    print(col)
    print("Unique Values:", df_clean[col].nunique())
    print(df_clean[col].value_counts(dropna=False).head(10))


ServiceArea
Unique Values: 748
ServiceArea
NYCBRO917    1684
HOUHOU281    1510
DALDAL214    1498
NYCMAN917    1182
APCFCH703     783
DALFTW817     782
SANSAN210     724
APCSIL301     670
SANAUS512     612
SFROAK510     605
Name: count, dtype: int64

CreditRating
Unique Values: 7
CreditRating
2-High       18993
1-Highest     8522
3-Good        8410
5-Low         6499
4-Medium      5357
7-Lowest      2114
6-VeryLow     1152
Name: count, dtype: int64

PrizmCode
Unique Values: 4
PrizmCode
Other       24655
Suburban    16378
Town         7589
Rural        2425
Name: count, dtype: int64

Occupation
Unique Values: 8
Occupation
Other           37637
Professional     8755
Crafts           1519
Clerical          986
Self              879
Retired           733
Student           381
Homemaker         157
Name: count, dtype: int64


In [43]:
df_clean["CustomerID"].nunique()

51047

## CustomerID Review

- CustomerID is a unique identifier.
- It does not represent customer behavior.
- It will be retained for traceability.
- It will be excluded from modeling.

In [44]:
df_clean.to_csv(
    "../data/processed/train_clean.csv",
    index=False
)

In [45]:
df_clean.shape

(51047, 61)

# Phase 4 Summary

Completed:

- Standardized hidden and encoded missing values.
- Applied appropriate imputation strategies.
- Preserved informative missingness through indicator variables.
- Reviewed leakage-risk features.
- Classified numerical and categorical variables.
- Identified CustomerID as a non-predictive identifier.

Output:

- Cleaned dataset with no missing values.
- Ready for feature engineering.